> **Adaptación de CAIS, Intro to ML Safety.** Este notebook adapta al español el notebook L2 del curso *Intro to ML Safety* (course.mlsafety.org) del Center for AI Safety. Reproduced with permission of Center for AI Safety, Inc. CAIS no respalda este curso ni tiene relación con él.
>
> Gran parte del código y del texto procede de la tarea *FullyConnectedNets* (assignment 2) de Stanford CS231n, publicada en github.com/cs231n/cs231n.github.io bajo licencia MIT (Copyright (c) 2015 Andrej Karpathy). Se reproduce con ese aviso. Fashion-MNIST: Zalando Research, MIT.
>
> Avisos y licencias completos: `NOTICE.md` en la raíz del repositorio.

# Redes totalmente conectadas

En este notebook implementamos redes totalmente conectadas con un enfoque modular. Para cada capa escribiremos una función `forward` (hacia delante) y otra `backward` (hacia atrás). La función `forward` recibe las entradas, los pesos y otros parámetros, y devuelve tanto una salida como un objeto `cache` con los datos que hacen falta en la pasada hacia atrás, así:

```python
def layer_forward(x, w):
  """ Recibe las entradas x y los pesos w """
  # Hace algunos cálculos ...
  z = # ... algún valor intermedio
  # Hace algunos cálculos más ...
  out = # la salida

  cache = (x, w, z, out) # Valores que necesitamos para calcular los gradientes

  return out, cache
```

La pasada hacia atrás recibe las derivadas que llegan desde arriba y el objeto `cache`, y devuelve los gradientes respecto a las entradas y a los pesos, así:

```python
def layer_backward(dout, cache):
  """
  Recibe la derivada de la pérdida respecto a las salidas y el cache,
  y calcula la derivada respecto a las entradas.
  """
  # Desempaqueta los valores del cache
  x, w, z, out = cache

  # Usa los valores del cache para calcular las derivadas
  dx = # Derivada de la pérdida respecto a x
  dw = # Derivada de la pérdida respecto a w

  return dx, dw
```

Después de implementar varias capas de esta manera, podremos combinarlas con facilidad para construir clasificadores con arquitecturas distintas.

Además de implementar redes totalmente conectadas de profundidad arbitraria, exploraremos diferentes reglas de actualización para la optimización, y presentaremos Dropout como regularizador y la normalización por lotes (Batch Normalization) como herramienta para optimizar redes profundas con más eficiencia.

> **Nota de la adaptación (curso de seguridad de la IA, lección m00/l04).** Es la adaptación al español del notebook de la lección L2 del curso *Intro to ML Safety*. El código es el del original; se han traducido los textos y los comentarios. Es un **esqueleto de ejercicio**: las celdas con `TODO` están vacías a propósito y hay que rellenarlas. Todo el cálculo va con NumPy en CPU, así que la GPU no hace falta para este notebook. Los bloques de construcción, el entrenamiento y la inferencia se explican en la lección m00/l03.

## Requisitos para ejecutar en local

El original corría en Google Colab: montaba el entorno y descargaba desde Google Drive un `.zip` con el material del ejercicio (`hw-fcnet-fashion-mnist.zip`, id de Drive `1mOerd6cQMKhOTmF5AoxQJ-RstCpirTM8`). Aquí se ejecuta en el equipo propio. Antes de empezar:

1. Un entorno de Python con Jupyter y estos paquetes: `numpy`, `matplotlib` y `torch`. Ejemplo: `pip install numpy matplotlib jupyter`, y PyTorch con la orden que da su web para tu sistema y tu versión de CUDA (con una GPU NVIDIA sirve, aunque este notebook no la usa). Comprueba que la versión de PyTorch admite la versión de Python que tengas.
2. La carpeta `deeplearning/` del ejercicio (con `data_utils.py`, `gradient_check.py`, `fast_layers.py` y `wrn_with_pen.py`) y los datos de Fashion-MNIST. Están dentro del `.zip` original: descárgalo tú a mano con el id de arriba y descomprímelo **al lado de este notebook**. Este curso no lo descarga ni lo incluye.
3. Abre el notebook desde la carpeta que contiene `deeplearning/`. La celda siguiente lo comprueba.

In [ ]:
# Requisitos para ejecución local (sustituye a las celdas de Colab que instalaban gdown,
# descargaban el zip y cambiaban de carpeta con os.chdir).
import importlib.util
import os

if importlib.util.find_spec("deeplearning") is None:
    raise FileNotFoundError(
        "No se encuentra la carpeta 'deeplearning/'. Descomprime el material del ejercicio "
        "(hw-fcnet-fashion-mnist.zip) junto a este notebook y ejecútalo desde esa carpeta. "
        "Carpeta actual: " + os.getcwd()
    )
print("Material del ejercicio encontrado en:", os.getcwd())

In [ ]:
# Como siempre, un poco de preparación

import time
import numpy as np
import matplotlib.pyplot as plt
import random
import torch

# para recargar módulos externos automáticamente
# véase http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

%matplotlib inline

In [ ]:
from deeplearning.data_utils import get_FMNIST_data
from deeplearning.gradient_check import eval_numerical_gradient, eval_numerical_gradient_array
from deeplearning.wrn_with_pen import WideResNet

seed = 7
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

plt.rcParams['figure.figsize'] = (10.0, 8.0) # tamaño por defecto de las gráficas
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

def rel_error(x, y):
    """ devuelve el error relativo """
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

In [ ]:
# Carga los datos de Fashion-MNIST.

data = get_FMNIST_data()
for k, v in data.items():
    print('%s: ' % k, v.shape)

# Capa afín: pasada hacia delante
Implementa la función `affine_forward`.

In [ ]:
def affine_forward(x, w, b):
    """
    Calcula la pasada hacia delante de una capa afín (totalmente conectada).

    La entrada x tiene forma (N, d_1, ..., d_k) y contiene un minilote de N
    ejemplos, donde cada ejemplo x[i] tiene forma (d_1, ..., d_k). Reordenaremos
    cada entrada como un vector de dimensión D = d_1 * ... * d_k, y luego lo
    transformaremos en un vector de salida de dimensión M.

    Entradas:
    - x: Un array de NumPy con los datos de entrada, de forma (N, d_1, ..., d_k)
    - w: Un array de NumPy con los pesos, de forma (D, M)
    - b: Un array de NumPy con los sesgos, de forma (M,)

    Devuelve una tupla con:
    - out: la salida, de forma (N, M)
    - cache: (x, w, b)
    """
    out = None
    #############################################################################
    # TODO: Implementa la pasada hacia delante afín. Guarda el resultado en out. #
    # Tendrás que reordenar la entrada en filas.                                #
    #############################################################################

    #############################################################################
    #                             FIN DE TU CÓDIGO                              #
    #############################################################################
    cache = (x, w, b)
    return out, cache

Cuando termines puedes probar tu implementación ejecutando lo siguiente:

In [ ]:
# Prueba la función affine_forward

num_inputs = 2
input_shape = (4, 5, 6)
output_dim = 3

input_size = num_inputs * np.prod(input_shape)
weight_size = output_dim * np.prod(input_shape)

x = np.linspace(-0.1, 0.5, num=input_size).reshape(num_inputs, *input_shape)
w = np.linspace(-0.2, 0.3, num=weight_size).reshape(np.prod(input_shape), output_dim)
b = np.linspace(-0.3, 0.1, num=output_dim)

out, _ = affine_forward(x, w, b)
correct_out = np.array([[ 1.49834967,  1.70660132,  1.91485297],
                        [ 3.25553199,  3.5141327,   3.77273342]])

# Compara tu salida con la nuestra. El error debería rondar 1e-9.
print('Probando la función affine_forward:')
print('diferencia: ', rel_error(out, correct_out))

# Capa afín: pasada hacia atrás
Ahora implementa la función `affine_backward` y prueba tu implementación con una comprobación numérica del gradiente.

In [ ]:
def affine_backward(dout, cache):
    """
    Calcula la pasada hacia atrás de una capa afín.

    Entradas:
    - dout: Derivada que llega de arriba, de forma (N, M)
    - cache: Tupla con:
      - x: Datos de entrada, de forma (N, d_1, ... d_k)
      - w: Pesos, de forma (D, M)

    Devuelve una tupla con:
    - dx: Gradiente respecto a x, de forma (N, d1, ..., d_k)
    - dw: Gradiente respecto a w, de forma (D, M)
    - db: Gradiente respecto a b, de forma (M,)
    """
    x, w, b = cache
    dx, dw, db = None, None, None
    #############################################################################
    # TODO: Implementa la pasada hacia atrás afín.                              #
    #############################################################################

    #############################################################################
    #                             FIN DE TU CÓDIGO                              #
    #############################################################################
    return dx, dw, db

In [ ]:
# Prueba la función affine_backward

x = np.random.randn(10, 2, 3)
w = np.random.randn(6, 5)
b = np.random.randn(5)
dout = np.random.randn(10, 5)

dx_num = eval_numerical_gradient_array(lambda x: affine_forward(x, w, b)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: affine_forward(x, w, b)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: affine_forward(x, w, b)[0], b, dout)

_, cache = affine_forward(x, w, b)
dx, dw, db = affine_backward(dout, cache)
# El error debería rondar 1e-10
print('Probando la función affine_backward:')
print('error de dx: ', rel_error(dx_num, dx))
print('error de dw: ', rel_error(dw_num, dw))
print('error de db: ', rel_error(db_num, db))

# Capa ReLU: pasada hacia delante
Implementa la pasada hacia delante de la función de activación ReLU en la función `relu_forward`.

In [ ]:
def relu_forward(x):
    """
    Calcula la pasada hacia delante de una capa de unidades lineales rectificadas (ReLU).

    Entrada:
    - x: Entradas, de cualquier forma

    Devuelve una tupla con:
    - out: Salida, con la misma forma que x
    - cache: x
    """
    out = None
    #############################################################################
    # TODO: Implementa la pasada hacia delante de ReLU.                         #
    #############################################################################

    #############################################################################
    #                             FIN DE TU CÓDIGO                              #
    #############################################################################
    cache = x
    return out, cache

Ahora prueba tu implementación con lo siguiente:

In [ ]:
# Prueba la función relu_forward

x = np.linspace(-0.5, 0.5, num=12).reshape(3, 4)

out, _ = relu_forward(x)
correct_out = np.array([[ 0.,          0.,          0.,          0.,        ],
                        [ 0.,          0.,          0.04545455,  0.13636364,],
                        [ 0.22727273,  0.31818182,  0.40909091,  0.5,       ]])

# Compara tu salida con la nuestra. El error debería rondar 1e-8
print('Probando la función relu_forward:')
print('diferencia: ', rel_error(out, correct_out))

# Capa ReLU: pasada hacia atrás
Ahora implementa la pasada hacia atrás de la función de activación ReLU en la función `relu_backward`. Ten en cuenta que ReLU no es derivable en 0, pero por lo general no nos preocupa: se le asigna por convención una derivada de 0 o de 1.

In [ ]:
def relu_backward(dout, cache):
    """
    Calcula la pasada hacia atrás de una capa de unidades lineales rectificadas (ReLU).

    Entrada:
    - dout: Derivadas que llegan de arriba, de cualquier forma
    - cache: Entrada x, con la misma forma que dout

    Devuelve:
    - dx: Gradiente respecto a x
    """
    dx, x = None, cache
    #############################################################################
    # TODO: Implementa la pasada hacia atrás de ReLU.                           #
    #############################################################################

    #############################################################################
    #                             FIN DE TU CÓDIGO                              #
    #############################################################################
    return dx

Prueba tu implementación con una comprobación numérica del gradiente.

In [ ]:
x = np.random.randn(10, 10)
dout = np.random.randn(*x.shape)

dx_num = eval_numerical_gradient_array(lambda x: relu_forward(x)[0], x, dout)

_, cache = relu_forward(x)
dx = relu_backward(dout, cache)

# El error debería rondar 1e-12
print('Probando la función relu_backward:')
print('error de dx: ', rel_error(dx_num, dx))

# Capas "sándwich"
Hay algunos patrones de capas que se usan mucho en las redes neuronales. Por ejemplo, después de una capa afín suele ir una no linealidad ReLU. Para facilitar estos patrones comunes, definimos por ti varias capas de conveniencia.

Por ahora, echa un vistazo a las funciones `affine_relu_forward` y `affine_relu_backward`, y ejecuta lo siguiente para comprobar numéricamente el gradiente de la pasada hacia atrás:

In [ ]:
from deeplearning.fast_layers import *


def affine_relu_forward(x, w, b):
    """
    Capa de conveniencia que aplica una transformación afín seguida de una ReLU

    Entradas:
    - x: Entrada de la capa afín
    - w, b: Pesos de la capa afín

    Devuelve una tupla con:
    - out: Salida de la ReLU
    - cache: Objeto que se pasa a la pasada hacia atrás
    """
    a, fc_cache = affine_forward(x, w, b)
    out, relu_cache = relu_forward(a)
    cache = (fc_cache, relu_cache)
    return out, cache


def affine_relu_backward(dout, cache):
    """
    Pasada hacia atrás de la capa de conveniencia afín-ReLU
    """
    fc_cache, relu_cache = cache
    da = relu_backward(dout, relu_cache)
    dx, dw, db = affine_backward(da, fc_cache)
    return dx, dw, db

In [ ]:
x = np.random.randn(2, 3, 4)
w = np.random.randn(12, 10)
b = np.random.randn(10)
dout = np.random.randn(2, 10)

out, cache = affine_relu_forward(x, w, b)
dx, dw, db = affine_relu_backward(dout, cache)

dx_num = eval_numerical_gradient_array(lambda x: affine_relu_forward(x, w, b)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: affine_relu_forward(x, w, b)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: affine_relu_forward(x, w, b)[0], b, dout)

print('Probando affine_relu_forward:')
print('error de dx: ', rel_error(dx_num, dx))
print('error de dw: ', rel_error(dw_num, dw))
print('error de db: ', rel_error(db_num, db))

# Capa de pérdida: Softmax
Aquí te damos la función de pérdida softmax que usaremos para entrenar nuestras redes neuronales profundas. Debes entender cómo funciona leyendo la implementación de abajo.

Puedes comprobar que la implementación es correcta ejecutando lo siguiente:

In [ ]:
def softmax_loss(x, y):
    """
    Calcula la pérdida y el gradiente para la clasificación softmax.

    Entradas:
    - x: Datos de entrada, de forma (N, C), donde x[i, j] es la puntuación de la
      clase j para la entrada i.
    - y: Vector de etiquetas, de forma (N,), donde y[i] es la etiqueta de x[i] y
      0 <= y[i] < C

    Devuelve una tupla con:
    - loss: Escalar con la pérdida
    - dx: Gradiente de la pérdida respecto a x
    """
    probs = np.exp(x - np.max(x, axis=1, keepdims=True))
    probs /= np.sum(probs, axis=1, keepdims=True)
    N = x.shape[0]
    loss = -np.sum(np.log(probs[np.arange(N), y])) / N
    dx = probs.copy()
    dx[np.arange(N), y] -= 1
    dx /= N
    return loss, dx

In [ ]:
num_classes, num_inputs = 10, 50
x = 0.001 * np.random.randn(num_inputs, num_classes)
y = np.random.randint(num_classes, size=num_inputs)

dx_num = eval_numerical_gradient(lambda x: softmax_loss(x, y)[0], x, verbose=False)
loss, dx = softmax_loss(x, y)

# Prueba la función softmax_loss. La pérdida debería ser 2,3 y el error de dx, 1e-8
print('\nProbando softmax_loss:')
print('pérdida: ', loss)
print('error de dx: ', rel_error(dx_num, dx))

# Red de dos capas

Completa la implementación de la clase `TwoLayerNet`. Esta clase servirá de modelo para las demás redes que implementarás en este ejercicio, así que léela con calma para entender bien su API.

In [ ]:
class TwoLayerNet(object):
    """
    Una red neuronal totalmente conectada de dos capas, con no linealidad ReLU y
    pérdida softmax, que usa un diseño modular de capas. Suponemos una dimensión
    de entrada D, una dimensión oculta H, y que clasificamos entre C clases.

    La arquitectura debe ser afín - relu - afín - softmax.

    Ten en cuenta que esta clase no implementa el descenso de gradiente; en su
    lugar, interactúa con un objeto Solver aparte, que se encarga de ejecutar
    la optimización.

    Los parámetros aprendibles del modelo se guardan en el diccionario
    self.params, que asocia nombres de parámetros con arrays de NumPy.
    """

    def __init__(self, input_dim=3 * 32 * 32, hidden_dim=100, num_classes=10,
                 weight_scale=1e-3, reg=0.0):
        """
        Inicializa una red nueva.

        Entradas:
        - input_dim: Un entero con el tamaño de la entrada
        - hidden_dim: Un entero con el tamaño de la capa oculta
        - num_classes: Un entero con el número de clases entre las que clasificar
        - dropout: Escalar entre 0 y 1 con la intensidad del dropout.
        - weight_scale: Escalar con la desviación típica de la inicialización
          aleatoria de los pesos.
        - reg: Escalar con la intensidad de la regularización L2.
        """
        self.params = {}
        self.reg = reg

        ############################################################################
        # TODO: Inicializa los pesos y los sesgos de la red de dos capas. Los      #
        # pesos deben salir de una gaussiana con desviación típica igual a         #
        # weight_scale, y los sesgos se inicializan a cero. Todos los pesos y      #
        # sesgos deben guardarse en el diccionario self.params, con las claves     #
        # 'W1' y 'b1' para los pesos y sesgos de la primera capa, y 'W2' y 'b2'    #
        # para los de la segunda.                                                  #
        ############################################################################

        ############################################################################
        #                             FIN DE TU CÓDIGO                             #
        ############################################################################

    def loss(self, X, y=None):
        """
        Calcula la pérdida y el gradiente para un minilote de datos.

        Entradas:
        - X: Array de datos de entrada, de forma (N, d_1, ..., d_k)
        - y: Array de etiquetas, de forma (N,). y[i] es la etiqueta de X[i].

        Devuelve:
        Si y es None, ejecuta una pasada hacia delante del modelo en modo de
        prueba y devuelve:
        - scores: Array de forma (N, C) con las puntuaciones de clasificación, donde
          scores[i, c] es la puntuación de la clase c para X[i].

        Si y no es None, ejecuta una pasada hacia delante y hacia atrás en modo de
        entrenamiento y devuelve una tupla con:
        - loss: Valor escalar con la pérdida
        - grads: Diccionario con las mismas claves que self.params, que asocia
          cada nombre de parámetro con el gradiente de la pérdida respecto a él.
        """
        scores = None
        ############################################################################
        # TODO: Implementa la pasada hacia delante de la red de dos capas,         #
        # calculando las puntuaciones de clase de X y guardándolas en la variable  #
        # scores.                                                                  #
        ############################################################################

        ############################################################################
        #                             FIN DE TU CÓDIGO                             #
        ############################################################################

        # Si y es None estamos en modo de prueba, así que devolvemos solo las puntuaciones
        if y is None:
            return scores

        loss, grads = 0, {}
        ############################################################################
        # TODO: Implementa la pasada hacia atrás de la red de dos capas. Guarda    #
        # la pérdida en la variable loss y los gradientes en el diccionario grads. #
        # Calcula la pérdida de los datos con softmax, y asegúrate de que grads[k] #
        # contiene los gradientes de self.params[k]. No olvides añadir la          #
        # regularización L2 sobre los pesos, pero no sobre los sesgos.             #
        #                                                                          #
        # NOTA: Para que tu implementación coincida con la nuestra y pase las      #
        # pruebas automáticas, asegúrate de que tu regularización L2 incluye un    #
        # factor 0,5 que simplifica la expresión del gradiente.                    #
        ############################################################################

        ############################################################################
        #                             FIN DE TU CÓDIGO                             #
        ############################################################################

        return loss, grads

Ejecuta la celda de abajo para probar tu implementación.

In [ ]:
N, D, H, C = 3, 5, 50, 7
X = np.random.randn(N, D)
y = np.random.randint(C, size=N)

std = 1e-2
model = TwoLayerNet(input_dim=D, hidden_dim=H, num_classes=C, weight_scale=std)

print ('Probando la inicialización ... ')
W1_std = abs(model.params['W1'].std() - std)
b1 = model.params['b1']
W2_std = abs(model.params['W2'].std() - std)
b2 = model.params['b2']
assert W1_std < std / 10, 'Los pesos de la primera capa no parecen correctos'
assert np.all(b1 == 0), 'Los sesgos de la primera capa no parecen correctos'
assert W2_std < std / 10, 'Los pesos de la segunda capa no parecen correctos'
assert np.all(b2 == 0), 'Los sesgos de la segunda capa no parecen correctos'

print ('Probando la pasada hacia delante en modo de prueba ... ')
model.params['W1'] = np.linspace(-0.7, 0.3, num=D*H).reshape(D, H)
model.params['b1'] = np.linspace(-0.1, 0.9, num=H)
model.params['W2'] = np.linspace(-0.3, 0.4, num=H*C).reshape(H, C)
model.params['b2'] = np.linspace(-0.9, 0.1, num=C)
X = np.linspace(-5.5, 4.5, num=N*D).reshape(D, N).T
scores = model.loss(X)
correct_scores = np.asarray(
  [[11.53165108,  12.2917344,   13.05181771,  13.81190102,  14.57198434, 15.33206765,  16.09215096],
   [12.05769098,  12.74614105,  13.43459113,  14.1230412,   14.81149128, 15.49994135,  16.18839143],
   [12.58373087,  13.20054771,  13.81736455,  14.43418138,  15.05099822, 15.66781506,  16.2846319 ]])
scores_diff = np.abs(scores - correct_scores).sum()
assert scores_diff < 1e-6, 'Problema en la pasada hacia delante en modo de prueba'

print ('Probando la pérdida de entrenamiento (sin regularización)')
y = np.asarray([0, 5, 1])
loss, grads = model.loss(X, y)
correct_loss = 3.4702243556
assert abs(loss - correct_loss) < 1e-10, 'Problema en la pérdida en modo de entrenamiento'

model.reg = 1.0
loss, grads = model.loss(X, y)
correct_loss = 26.5948426952
assert abs(loss - correct_loss) < 1e-10, 'Problema en la pérdida de regularización'

for reg in [0.0, 0.7]:
    print ('Ejecutando la comprobación numérica del gradiente con reg = ', reg)
    model.reg = reg
    loss, grads = model.loss(X, y)

    for name in sorted(grads):
        f = lambda _: model.loss(X, y)[0]
        grad_num = eval_numerical_gradient(f, model.params[name], verbose=False)
        print ('%s error relativo: %.2e' % (name, rel_error(grad_num, grads[name])))

# Solver
Siguiendo un diseño modular, en este ejercicio hemos separado la lógica de entrenamiento de los modelos en una clase aparte.

Abre la celda de abajo, que contiene una copia de la clase Solver, y léela para familiarizarte con su API. Después, usa una instancia de `Solver` para entrenar una `TwoLayerNet` que alcance al menos un `50%` de precisión en el conjunto de validación.

In [ ]:
# De Stanford CS231n, assignment 2 (MIT, Copyright (c) 2015 Andrej Karpathy). Adaptado.
class Solver(object):
    """
    Un Solver encapsula toda la lógica necesaria para entrenar modelos de
    clasificación. El Solver ejecuta el descenso de gradiente estocástico con
    distintas reglas de actualización, definidas en optim.py.

    El solver acepta datos y etiquetas de entrenamiento y de validación, para
    poder comprobar cada cierto tiempo la precisión en ambos conjuntos y vigilar
    el sobreajuste.

    Para entrenar un modelo, primero construyes una instancia de Solver pasándole
    el modelo, el conjunto de datos y varias opciones (tasa de aprendizaje,
    tamaño del lote, etc.) al constructor. Luego llamas al método train() para
    ejecutar el procedimiento de optimización y entrenar el modelo.

    Cuando train() termina, model.params contiene los parámetros que mejor
    resultado dieron en validación a lo largo del entrenamiento. Además, la
    variable de instancia solver.loss_history tendrá una lista con todas las
    pérdidas que aparecieron durante el entrenamiento, y las variables de
    instancia solver.train_acc_history y solver.val_acc_history serán listas con
    las precisiones en entrenamiento y validación al final de cada época.

    Un uso de ejemplo podría ser así:

    data = {
      'X_train': # datos de entrenamiento
      'y_train': # etiquetas de entrenamiento
      'X_val': # datos de validación
      'X_train': # etiquetas de validación
    }
    model = MyAwesomeModel(hidden_size=100, reg=10)
    solver = Solver(model, data,
                    update_rule='sgd',
                    optim_config={
                      'learning_rate': 1e-3,
                    },
                    lr_decay=0.95,
                    num_epochs=10, batch_size=100,
                    print_every=100)
    solver.train()


    Un Solver trabaja con un objeto modelo que debe cumplir esta API:

    - model.params debe ser un diccionario que asocie nombres de parámetros
      (cadenas) con arrays de NumPy con los valores de los parámetros.

    - model.loss(X, y) debe ser una función que calcule la pérdida y los
      gradientes en tiempo de entrenamiento, y las puntuaciones de clasificación
      en tiempo de prueba, con estas entradas y salidas:

      Entradas:
      - X: Array con un minilote de datos de entrada, de forma (N, d_1, ..., d_k)
      - y: Array de etiquetas, de forma (N,), donde y[i] es la etiqueta de X[i].

      Devuelve:
      Si y es None, ejecuta una pasada hacia delante en modo de prueba y devuelve:
      - scores: Array de forma (N, C) con las puntuaciones de clasificación de X,
        donde scores[i, c] es la puntuación de la clase c para X[i].

      Si y no es None, ejecuta una pasada hacia delante y hacia atrás en modo de
      entrenamiento y devuelve una tupla con:
      - loss: Escalar con la pérdida
      - grads: Diccionario con las mismas claves que self.params, que asocia
        cada nombre de parámetro con el gradiente de la pérdida respecto a él.
    """

    def __init__(self, model, data, **kwargs):
        """
        Construye una instancia nueva de Solver.

        Argumentos obligatorios:
        - model: Un objeto modelo que cumpla la API descrita arriba
        - data: Un diccionario con datos de entrenamiento y validación:
          'X_train': Array de forma (N_train, d_1, ..., d_k) con las imágenes de entrenamiento
          'X_val': Array de forma (N_val, d_1, ..., d_k) con las imágenes de validación
          'y_train': Array de forma (N_train,) con las etiquetas de las imágenes de entrenamiento
          'y_val': Array de forma (N_val,) con las etiquetas de las imágenes de validación

        Argumentos opcionales:
        - update_rule: Una cadena con el nombre de una regla de actualización de
          optim.py. Por defecto es 'sgd'.
        - optim_config: Un diccionario con hiperparámetros que se pasarán a la
          regla de actualización elegida. Cada regla necesita hiperparámetros
          distintos (véase optim.py), pero todas requieren un parámetro
          'learning_rate', así que siempre debe estar presente.
        - lr_decay: Un escalar para el decaimiento de la tasa de aprendizaje; tras
          cada época, la tasa de aprendizaje se multiplica por este valor.
        - batch_size: Tamaño de los minilotes usados para calcular la pérdida y el
          gradiente durante el entrenamiento.
        - num_epochs: Número de épocas que dura el entrenamiento.
        - print_every: Entero; la pérdida de entrenamiento se imprime cada
          print_every iteraciones.
        - verbose: Booleano; si es falso, no se imprime nada durante el
          entrenamiento.
        """
        self.model = model
        self.X_train = data['X_train']
        self.y_train = data['y_train']
        self.X_val = data['X_val']
        self.y_val = data['y_val']

        # Desempaqueta los argumentos con nombre
        self.update_rule = kwargs.pop('update_rule', 'sgd')
        self.optim_config = kwargs.pop('optim_config', {})
        self.lr_decay = kwargs.pop('lr_decay', 1.0)
        self.batch_size = kwargs.pop('batch_size', 100)
        self.num_epochs = kwargs.pop('num_epochs', 10)

        self.print_every = kwargs.pop('print_every', 10)
        self.verbose = kwargs.pop('verbose', True)

        # Lanza un error si sobran argumentos con nombre
        if len(kwargs) > 0:
            extra = ', '.join('"%s"' % k for k in kwargs.keys())
            raise ValueError('Argumentos no reconocidos %s' % extra)

        # Comprueba que la regla de actualización existe y sustituye el nombre
        # (cadena) por la función real
        self.update_rule = globals()[self.update_rule]

        self._reset()

    def _reset(self):
        """
        Prepara algunas variables de contabilidad para la optimización. No
        llames a este método a mano.
        """
        # Prepara algunas variables de contabilidad
        self.epoch = 0
        self.best_val_acc = 0
        self.best_params = {}
        self.loss_history = []
        self.train_acc_history = []
        self.val_acc_history = []

        # Hace una copia profunda de optim_config para cada parámetro
        self.optim_configs = {}
        for p in self.model.params:
            d = {k: v for k, v in self.optim_config.items()}
            self.optim_configs[p] = d

    def _step(self):
        """
        Hace una sola actualización de gradiente. Lo llama train() y no debe
        llamarse a mano.
        """
        # Construye un minilote de datos de entrenamiento
        num_train = self.X_train.shape[0]
        batch_mask = np.random.choice(num_train, self.batch_size)
        X_batch = self.X_train[batch_mask]
        y_batch = self.y_train[batch_mask]

        # Calcula la pérdida y el gradiente
        loss, grads = self.model.loss(X_batch, y_batch)
        self.loss_history.append(loss)

        # Actualiza los parámetros
        for p, w in self.model.params.items():
            dw = grads[p]
            config = self.optim_configs[p]
            next_w, next_config = self.update_rule(w, dw, config)
            self.model.params[p] = next_w
            self.optim_configs[p] = next_config


    def check_accuracy(self, X, y, num_samples=None, batch_size=100):
        """
        Comprueba la precisión del modelo con los datos dados.

        Entradas:
        - X: Array de datos, de forma (N, d_1, ..., d_k)
        - y: Array de etiquetas, de forma (N,)
        - num_samples: Si no es None, submuestrea los datos y prueba el modelo
          solo con num_samples puntos.
        - batch_size: Divide X e y en lotes de este tamaño para no gastar
          demasiada memoria.

        Devuelve:
        - acc: Escalar con la fracción de instancias que el modelo clasificó
          correctamente.
        """

        # Submuestrea los datos si hace falta
        N = X.shape[0]
        if num_samples is not None and N > num_samples:
            mask = np.random.choice(N, num_samples)
            N = num_samples
            X = X[mask]
            y = y[mask]

        # Calcula las predicciones por lotes
        num_batches = N // batch_size
        if N % batch_size != 0:
            num_batches += 1
        y_pred = []
        for i in range(num_batches):
            start = i * batch_size
            end = (i + 1) * batch_size
            scores = self.model.loss(X[start:end])
            y_pred.append(np.argmax(scores, axis=1))
        y_pred = np.hstack(y_pred)
        acc = np.mean(y_pred == y)

        return acc

    def train(self):
        """
        Ejecuta la optimización para entrenar el modelo.
        """
        num_train = self.X_train.shape[0]
        iterations_per_epoch = max(num_train // self.batch_size, 1)
        num_iterations = self.num_epochs * iterations_per_epoch

        for t in range(num_iterations):
            self._step()

            # Imprime la pérdida de entrenamiento, si toca
            if self.verbose and t % self.print_every == 0:
                print('(Iteración %d / %d) pérdida: %f' % (t + 1, num_iterations, self.loss_history[-1]))

            # Al final de cada época, incrementa el contador de épocas y aplica
            # el decaimiento a la tasa de aprendizaje.
            epoch_end = (t + 1) % iterations_per_epoch == 0
            if epoch_end:
                self.epoch += 1
                for k in self.optim_configs:
                    self.optim_configs[k]['learning_rate'] *= self.lr_decay

            # Comprueba la precisión en entrenamiento y validación en la primera
            # iteración, en la última y al final de cada época.
            first_it = (t == 0)
            last_it = (t == num_iterations + 1)
            if first_it or last_it or epoch_end:
                train_acc = self.check_accuracy(self.X_train, self.y_train,
                                                num_samples=1000)
                val_acc = self.check_accuracy(self.X_val, self.y_val)
                self.train_acc_history.append(train_acc)
                self.val_acc_history.append(val_acc)

                if self.verbose:
                    print('(Época %d / %d) precisión en entrenamiento: %f; val_acc: %f' % (
                        self.epoch, self.num_epochs, train_acc, val_acc))

                # Guarda el mejor modelo hasta el momento
                if val_acc > self.best_val_acc:
                    self.best_val_acc = val_acc
                    self.best_params = {}
                    for k, v in self.model.params.items():
                        self.best_params[k] = v.copy()

        # Al terminar el entrenamiento, pone los mejores parámetros en el modelo
        self.model.params = self.best_params

In [ ]:
model = TwoLayerNet()
solver = None

##############################################################################
# TODO: Usa una instancia de Solver para entrenar una TwoLayerNet que       #
# alcance al menos un 50% de precisión en el conjunto de validación.         #
##############################################################################

##############################################################################
#                             FIN DE TU CÓDIGO                               #
##############################################################################

> **Nota de la adaptación.** `TwoLayerNet()` usa por defecto `input_dim=3 * 32 * 32`, que corresponde a imágenes de color de 32 × 32. El resto del notebook trabaja con imágenes de `1 * 28 * 28` (es el valor por defecto de `FullyConnectedNet`), así que al construir la red puede que tengas que pasar el `input_dim` adecuado a los datos que cargaste.

In [ ]:
# Ejecuta esta celda para visualizar la pérdida de entrenamiento y la precisión en
# entrenamiento y validación, y para guardar el registro del experimento y entregarlo.

plt.subplot(2, 1, 1)
plt.title('Pérdida de entrenamiento')
plt.plot(solver.loss_history, 'o')
plt.xlabel('Iteración')

plt.subplot(2, 1, 2)
plt.title('Precisión')
plt.plot(solver.train_acc_history, '-o', label='entrenamiento')
plt.plot(solver.val_acc_history, '-o', label='validación')
plt.plot([0.5] * len(solver.val_acc_history), 'k--')
plt.xlabel('Época')
plt.legend(loc='lower right')
plt.gcf().set_size_inches(15, 12)
plt.show()

# Red multicapa
A continuación implementarás una red totalmente conectada con un número arbitrario de capas ocultas.

Implementa la inicialización, la pasada hacia delante y la pasada hacia atrás. Por ahora no te preocupes de implementar dropout ni normalización por lotes. Los veremos en la próxima sesión (sobre redes neuronales convolucionales) y dejamos su implementación aquí como un ejercicio divertido.

> **Nota de la adaptación.** En este curso, las piezas de construcción como dropout y la normalización se explican en la lección m00/l03.

In [ ]:
class FullyConnectedNet(object):
    """
    Una red neuronal totalmente conectada con un número arbitrario de capas
    ocultas, no linealidades ReLU y pérdida softmax. También implementará
    dropout y normalización por lotes como opciones. Para una red de L capas,
    la arquitectura será

    {afín - [batch norm] - relu - [dropout]} x (L - 1) - afín - softmax

    donde la normalización por lotes y el dropout son opcionales, y el bloque
    {...} se repite L - 1 veces.

    Igual que en TwoLayerNet, los parámetros aprendibles se guardan en el
    diccionario self.params y se aprenderán con la clase Solver.
    """

    def __init__(self, hidden_dims, input_dim=1 * 28 * 28, num_classes=10,
                 dropout=0, use_batchnorm=False, reg=0.0,
                 weight_scale=1e-2, dtype=np.float32, seed=None):
        """
        Inicializa una FullyConnectedNet nueva.

        Entradas:
        - hidden_dims: Una lista de enteros con el tamaño de cada capa oculta.
        - input_dim: Un entero con el tamaño de la entrada.
        - num_classes: Un entero con el número de clases entre las que clasificar.
        - dropout: Escalar entre 0 y 1 con la intensidad del dropout. Si dropout=0,
          la red no debe usar dropout en absoluto.
        - use_batchnorm: Si la red debe usar o no normalización por lotes.
        - reg: Escalar con la intensidad de la regularización L2.
        - weight_scale: Escalar con la desviación típica de la inicialización
          aleatoria de los pesos.
        - dtype: Un objeto de tipo de datos de NumPy; todos los cálculos se harán
          con este tipo. float32 es más rápido pero menos preciso, así que debes
          usar float64 para la comprobación numérica del gradiente.
        - seed: Si no es None, pasa esta semilla aleatoria a las capas de dropout.
          Así las capas de dropout serán deterministas y podremos comprobar el
          gradiente del modelo.
        """
        self.use_batchnorm = use_batchnorm
        self.use_dropout = dropout > 0
        self.reg = reg
        self.num_layers = 1 + len(hidden_dims)
        self.dtype = dtype
        self.params = {}

        ############################################################################
        # TODO: Inicializa los parámetros de la red y guarda todos los valores en  #
        # el diccionario self.params. Guarda los pesos y sesgos de la primera capa #
        # en W1 y b1; los de la segunda en W2 y b2, etc. Los pesos deben salir de  #
        # una distribución normal con desviación típica igual a weight_scale y los #
        # sesgos deben inicializarse a cero.                                       #
        #                                                                          #
        # Con normalización por lotes, guarda los parámetros de escala y           #
        # desplazamiento de la primera capa en gamma1 y beta1; los de la segunda   #
        # en gamma2 y beta2, etc. Los parámetros de escala deben inicializarse a   #
        # uno y los de desplazamiento a cero.                                      #
        ############################################################################

        ############################################################################
        #                             FIN DE TU CÓDIGO                             #
        ############################################################################

        # Con dropout necesitamos pasar un diccionario dropout_param a cada capa de
        # dropout para que la capa conozca la probabilidad de dropout y el modo
        # (entrenamiento / prueba). Puedes pasar el mismo dropout_param a todas las
        # capas de dropout.
        self.dropout_param = {}
        if self.use_dropout:
            self.dropout_param = {'mode': 'train', 'p': dropout}
            if seed is not None:
                self.dropout_param['seed'] = seed

        # Con normalización por lotes necesitamos llevar la cuenta de las medias y
        # varianzas móviles, así que pasamos un objeto especial bn_param a cada
        # capa de normalización por lotes. Debes pasar self.bn_params[0] a la
        # pasada hacia delante de la primera capa de normalización por lotes,
        # self.bn_params[1] a la de la segunda capa, etc.
        self.bn_params = []
        if self.use_batchnorm:
            self.bn_params = [{'mode': 'train'} for i in range(self.num_layers)]

        # Convierte todos los parámetros al tipo de datos correcto
        for k, v in self.params.items():
            self.params[k] = v.astype(dtype)

    def loss(self, X, y=None):
        """
        Calcula la pérdida y el gradiente de la red totalmente conectada.

        Entrada / salida: igual que en TwoLayerNet más arriba.
        """
        X = X.astype(self.dtype)
        mode = 'test' if y is None else 'train'

        # Fija el modo de entrenamiento/prueba en los parámetros de batchnorm y
        # de dropout, porque se comportan de forma distinta al entrenar y al probar.
        if self.dropout_param is not None:
            self.dropout_param['mode'] = mode
        if self.use_batchnorm:
            for bn_param in self.bn_params:
                bn_param[mode] = mode

        scores = None
        ############################################################################
        # TODO: Implementa la pasada hacia delante de la red totalmente conectada, #
        # calculando las puntuaciones de clase de X y guardándolas en la variable  #
        # scores.                                                                  #
        #                                                                          #
        # Con dropout, tendrás que pasar self.dropout_param a cada pasada hacia    #
        # delante de dropout.                                                      #
        #                                                                          #
        # Con normalización por lotes, tendrás que pasar self.bn_params[0] a la    #
        # pasada hacia delante de la primera capa de normalización por lotes,      #
        # self.bn_params[1] a la de la segunda capa, etc.                          #
        ############################################################################

        ############################################################################
        #                             FIN DE TU CÓDIGO                             #
        ############################################################################
        # En modo de prueba, devuelve ya
        if mode == 'test':
            return scores

        loss, grads = 0.0, {}
        ############################################################################
        # TODO: Implementa la pasada hacia atrás de la red totalmente conectada.   #
        # Guarda la pérdida en la variable loss y los gradientes en el diccionario #
        # grads. Calcula la pérdida de los datos con softmax, y asegúrate de que   #
        # grads[k] contiene los gradientes de self.params[k]. No olvides añadir la #
        # regularización L2 sobre los pesos, pero no sobre los sesgos.             #
        #                                                                          #
        # Con normalización por lotes, no hace falta regularizar los parámetros de #
        # escala y desplazamiento.                                                 #
        #                                                                          #
        # NOTA: Para que tu implementación coincida con la nuestra y pase las      #
        # pruebas automáticas, asegúrate de que tu regularización L2 incluye un    #
        # factor 0,5 que simplifica la expresión del gradiente.                    #
        ############################################################################

        ############################################################################
        #                             FIN DE TU CÓDIGO                             #
        ############################################################################

        return loss, grads

## Pérdida inicial y comprobación del gradiente

Como comprobación de cordura, ejecuta lo siguiente para revisar la pérdida inicial y comprobar numéricamente el gradiente de la red, con y sin regularización. ¿Te parecen razonables las pérdidas iniciales?

En la comprobación del gradiente deberías ver errores de alrededor de 1e-6 o menos.

In [ ]:
N, D, H1, H2, C = 2, 15, 20, 30, 10
X = np.random.randn(N, D)
y = np.random.randint(C, size=(N,))

for reg in [0, 3.14]:
    print ('Ejecutando la comprobación con reg = ', reg)
    model = FullyConnectedNet([H1, H2], input_dim=D, num_classes=C,
                              reg=reg, weight_scale=5e-2, dtype=np.float64)

    loss, grads = model.loss(X, y)
    print ('Pérdida inicial: ', loss)

    for name in sorted(grads):
        f = lambda _: model.loss(X, y)[0]
        grad_num = eval_numerical_gradient(f, model.params[name], verbose=False, h=1e-5)
        print ('%s error relativo: %.2e' % (name, rel_error(grad_num, grads[name])))

Como otra comprobación de cordura, asegúrate de que eres capaz de sobreajustar un conjunto pequeño de 50 imágenes. Primero probaremos con una red de tres capas con 100 unidades en cada capa oculta. Tendrás que ajustar la tasa de aprendizaje y la escala de inicialización, pero deberías poder sobreajustar y alcanzar un 100% de precisión en entrenamiento en 20 épocas.

In [ ]:
# TODO: Usa una red de tres capas para sobreajustar 50 ejemplos de entrenamiento.

num_train = 50
small_data = {
  'X_train': data['X_train'][:num_train],
  'y_train': data['y_train'][:num_train],
  'X_val': data['X_val'],
  'y_val': data['y_val'],
}

############################################################################
# TODO: Ajusta estos parámetros para llegar al 100% de precisión en        #
# entrenamiento en 20 épocas.                                              #
############################################################################
weight_scale = None
learning_rate = None
############################################################################
#                             FIN DE TU CÓDIGO                             #
############################################################################

model = FullyConnectedNet([100, 100],
              weight_scale=weight_scale, dtype=np.float64)
solver = Solver(model, small_data,
                print_every=10, num_epochs=20, batch_size=25,
                update_rule='sgd',
                optim_config={
                  'learning_rate': learning_rate,
                }
         )
solver.train()

plt.plot(solver.loss_history, 'o')
plt.title('Historial de la pérdida de entrenamiento')
plt.xlabel('Iteración')
plt.ylabel('Pérdida de entrenamiento')
plt.show()

Ahora prueba con una red de cinco capas con 100 unidades en cada capa para sobreajustar 50 ejemplos de entrenamiento. De nuevo tendrás que ajustar la tasa de aprendizaje y la inicialización de los pesos, pero deberías poder alcanzar un 100% de precisión en entrenamiento en 20 épocas.

In [ ]:
## TODO: Usa una red de cinco capas para sobreajustar 50 ejemplos de entrenamiento.

num_train = 50
small_data = {
  'X_train': data['X_train'][:num_train],
  'y_train': data['y_train'][:num_train],
  'X_val': data['X_val'],
  'y_val': data['y_val'],
}

############################################################################
# TODO: Ajusta estos parámetros para llegar al 100% de precisión en        #
# entrenamiento en 20 épocas.                                              #
############################################################################

############################################################################
#                             FIN DE TU CÓDIGO                             #
############################################################################

model = FullyConnectedNet([100, 100, 100, 100],
                weight_scale=weight_scale, dtype=np.float64)
solver = Solver(model, small_data,
                print_every=10, num_epochs=20, batch_size=25,
                update_rule='sgd',
                optim_config={
                  'learning_rate': learning_rate,
                }
         )
solver.train()

plt.plot(solver.loss_history, 'o')
plt.title('Historial de la pérdida de entrenamiento')
plt.xlabel('Iteración')
plt.ylabel('Pérdida de entrenamiento')
plt.show()

# Pregunta en línea:
¿Notaste algo sobre la dificultad comparada de entrenar la red de tres capas frente a la de cinco capas?

# Respuesta:

Escribe aquí tu respuesta

# Reglas de actualización
Hasta ahora hemos usado el descenso de gradiente estocástico (SGD) simple como regla de actualización. Reglas de actualización más sofisticadas pueden facilitar el entrenamiento de redes profundas. Implementaremos algunas de las más usadas y las compararemos con el SGD simple.

# SGD con momento
El descenso de gradiente estocástico con momento (momentum) es una regla de actualización muy usada que suele hacer que las redes profundas converjan más deprisa que el descenso de gradiente estocástico simple.

Lee la documentación siguiente para asegurarte de que entiendes la API que hemos preparado para nuestros optimizadores.

#### Documentación de la API de los optimizadores

Hemos implementado varias reglas de actualización de primer orden que se usan
mucho para entrenar redes neuronales. Cada regla de actualización recibe los
pesos actuales y el gradiente de la pérdida respecto a esos pesos, y produce
el siguiente conjunto de pesos. Todas las reglas de actualización tienen la
misma interfaz:

```python
def update(w, dw, config=None):
```

Entradas:
  - w: Un array de NumPy con los pesos actuales.
  - dw: Un array de NumPy con la misma forma que w y el gradiente de la
    pérdida respecto a w.
  - config: Un diccionario con valores de hiperparámetros como la tasa de
    aprendizaje, el momento, etc. Si la regla de actualización necesita guardar
    valores a lo largo de muchas iteraciones, config también contendrá esos
    valores guardados.

Devuelve:
  - next_w: El siguiente punto tras la actualización.
  - config: El diccionario config que se pasará a la siguiente iteración de
    la regla de actualización.

NOTA: En la mayoría de las reglas de actualización, la tasa de aprendizaje por
defecto probablemente no funcionará bien; en cambio, los valores por defecto
de los demás hiperparámetros deberían funcionar bien en muchos problemas
distintos.

Por eficiencia, las reglas de actualización pueden hacer actualizaciones en el
mismo sitio, modificando w y haciendo next_w igual a w.

Implementa la regla de actualización SGD con momento en la función `sgd_momentum` y ejecuta lo siguiente para comprobar tu implementación. Deberías ver errores menores que 1e-8.

In [ ]:
# De Stanford CS231n, assignment 2 (MIT, Copyright (c) 2015 Andrej Karpathy). Adaptado.
def sgd(w, dw, config=None):
    """
    Ejecuta el descenso de gradiente estocástico simple.

    formato de config:
    - learning_rate: Escalar con la tasa de aprendizaje.
    """
    if config is None: config = {}
    config.setdefault('learning_rate', 1e-2)

    w -= config['learning_rate'] * dw
    return w, config


def sgd_momentum(w, dw, config=None):
    """
    Ejecuta el descenso de gradiente estocástico con momento.

    formato de config:
    - learning_rate: Escalar con la tasa de aprendizaje.
    - momentum: Escalar entre 0 y 1 con el valor del momento.
      Con momentum = 0 se reduce a sgd.
    - velocity: Un array de NumPy con la misma forma que w y dw, que guarda una
      media móvil de los gradientes.
    """
    if config is None: config = {}
    config.setdefault('learning_rate', 1e-2)
    config.setdefault('momentum', 0.9)
    v = config.get('velocity', np.zeros_like(w))

    next_w = None
    #############################################################################
    # TODO: Implementa la fórmula de actualización con momento tal como se dio  #
    # en clase. Guarda el valor actualizado en la variable next_w. También      #
    # debes usar y actualizar la velocidad v.                                   #
    #############################################################################

    #############################################################################
    #                             FIN DE TU CÓDIGO                              #
    #############################################################################
    config['velocity'] = v

    return next_w, config

In [ ]:
N, D = 4, 5
w = np.linspace(-0.4, 0.6, num=N*D).reshape(N, D)
dw = np.linspace(0.5, 1.5, num=N*D).reshape(N, D)
v = np.linspace(0.6, 0.9, num=N*D).reshape(N, D)

config = {'learning_rate': 1e-1, 'velocity': v}
next_w, _ = sgd_momentum(w, dw, config=config)

expected_next_w = np.asarray([
 [-0.504,      -0.45805263, -0.41210526, -0.36615789, -0.32021053],
 [-0.27426316, -0.22831579, -0.18236842, -0.13642105, -0.09047368],
 [-0.04452632,  0.00142105,  0.04736842,  0.09331579,  0.13926316],
 [ 0.18521053,  0.23115789,  0.27710526,  0.32305263,  0.369     ]])

expected_velocity = np.asarray([
 [1.04,       1.10684211, 1.17368421, 1.24052632, 1.30736842],
 [1.37421053, 1.44105263, 1.50789474, 1.57473684, 1.64157895],
 [1.70842105, 1.77526316, 1.84210526, 1.90894737, 1.97578947],
 [2.04263158, 2.10947368, 2.17631579, 2.24315789, 2.31      ]])

print ('error de next_w: ', rel_error(next_w, expected_next_w))
print ('error de velocity: ', rel_error(expected_velocity, config['velocity']))

Cuando lo hayas hecho, ejecuta lo siguiente para entrenar una red de seis capas con SGD y con SGD con momento. Deberías ver que la regla SGD con momento converge un poco más deprisa.

In [ ]:
num_train = 4000
small_data = {
  'X_train': data['X_train'][:num_train],
  'y_train': data['y_train'][:num_train],
  'X_val': data['X_val'],
  'y_val': data['y_val'],
}

solvers = {}

for update_rule in ['sgd', 'sgd_momentum']:
    print ('ejecutando con ', update_rule)
    model = FullyConnectedNet([100, 100, 100, 100, 100], weight_scale=5e-2)

    solver = Solver(model, small_data,
                  num_epochs=5, batch_size=100,
                  update_rule=update_rule,
                  optim_config={
                    'learning_rate': 1e-2,
                  },
                  verbose=True)
    solvers[update_rule] = solver
    solver.train()
    print

plt.subplot(3, 1, 1)
plt.title('Pérdida de entrenamiento')
plt.xlabel('Iteración')

plt.subplot(3, 1, 2)
plt.title('Precisión en entrenamiento')
plt.xlabel('Época')

plt.subplot(3, 1, 3)
plt.title('Precisión en validación')
plt.xlabel('Época')

for update_rule, solver in solvers.items():
    plt.subplot(3, 1, 1)
    plt.plot(solver.loss_history, 'o', label=update_rule)

    plt.subplot(3, 1, 2)
    plt.plot(solver.train_acc_history, '-o', label=update_rule)

    plt.subplot(3, 1, 3)
    plt.plot(solver.val_acc_history, '-o', label=update_rule)

for i in [1, 2, 3]:
    plt.subplot(3, 1, i)
    plt.legend(loc='upper center', ncol=4)
plt.gcf().set_size_inches(15, 15)
plt.show()

# Adam
Adam[1] es una regla de actualización que fija tasas de aprendizaje por parámetro usando una media móvil de los segundos momentos de los gradientes.

[1] Diederik Kingma y Jimmy Ba, "Adam: A Method for Stochastic Optimization", ICLR 2015.

Implementa la regla de actualización de Adam en la función `adam` y comprueba tu implementación con las pruebas de abajo.

In [ ]:
# De Stanford CS231n, assignment 2 (MIT, Copyright (c) 2015 Andrej Karpathy). Adaptado.
def adam(x, dx, config=None):
    """
    Usa la regla de actualización de Adam, que incorpora medias móviles del
    gradiente y de su cuadrado, y un término de corrección del sesgo.

    formato de config:
    - learning_rate: Escalar con la tasa de aprendizaje.
    - beta1: Tasa de decaimiento de la media móvil del gradiente (primer momento).
    - beta2: Tasa de decaimiento de la media móvil del cuadrado del gradiente (segundo momento).
    - epsilon: Escalar pequeño para suavizar y evitar dividir entre cero.
    - m: Media móvil del gradiente.
    - v: Media móvil del cuadrado del gradiente.
    - t: Número de iteración.
    """
    if config is None: config = {}
    config.setdefault('learning_rate', 1e-3)
    config.setdefault('beta1', 0.9)
    config.setdefault('beta2', 0.999)
    config.setdefault('epsilon', 1e-8)
    config.setdefault('m', np.zeros_like(x))
    config.setdefault('v', np.zeros_like(x))
    config.setdefault('t', 0)

    next_x = None
    #############################################################################
    # TODO: Implementa la fórmula de actualización de Adam, guardando el        #
    # siguiente valor de x en la variable next_x. No olvides actualizar las     #
    # variables m, v y t guardadas en config, y usar el escalar epsilon para    #
    # evitar dividir entre cero.                                                #
    #############################################################################

    #############################################################################
    #                             FIN DE TU CÓDIGO                              #
    #############################################################################

    return next_x, config

In [ ]:
# Prueba la implementación de Adam; deberías ver errores de alrededor de 1e-7 o menos

N, D = 4, 5
w = np.linspace(-0.4, 0.6, num=N*D).reshape(N, D)
dw = np.linspace(-0.6, 0.4, num=N*D).reshape(N, D)
m = np.linspace(0.6, 0.9, num=N*D).reshape(N, D)
v = np.linspace(0.7, 0.5, num=N*D).reshape(N, D)

config = {'learning_rate': 1e-2, 'm': m, 'v': v, 't': 5}
next_w, _ = adam(w, dw, config=config)

expected_next_w = np.asarray([
  [-0.40094747, -0.34836187, -0.29577703, -0.24319299, -0.19060977],
  [-0.1380274,  -0.08544591, -0.03286534,  0.01971428,  0.0722929],
  [ 0.1248705,   0.17744702,  0.23002243,  0.28259667,  0.33516969],
  [ 0.38774145,  0.44031188,  0.49288093,  0.54544852,  0.59801459]])
expected_v = np.asarray([
  [ 0.69966,     0.68908382,  0.67851319,  0.66794809,  0.65738853,],
  [ 0.64683452,  0.63628604,  0.6257431,   0.61520571,  0.60467385,],
  [ 0.59414753,  0.58362676,  0.57311152,  0.56260183,  0.55209767,],
  [ 0.54159906,  0.53110598,  0.52061845,  0.51013645,  0.49966,   ]])
expected_m = np.asarray([
  [ 0.48,        0.49947368,  0.51894737,  0.53842105,  0.55789474],
  [ 0.57736842,  0.59684211,  0.61631579,  0.63578947,  0.65526316],
  [ 0.67473684,  0.69421053,  0.71368421,  0.73315789,  0.75263158],
  [ 0.77210526,  0.79157895,  0.81105263,  0.83052632,  0.85      ]])
expected_t = 6

print ('error de next_w: ', rel_error(expected_next_w, next_w))
print ('error de v: ', rel_error(expected_v, config['v']))
print ('error de m: ', rel_error(expected_m, config['m']))
print ('error de t: ', rel_error(expected_t, config['t']))

# ¡Entrena un buen modelo!
Entrena el mejor modelo totalmente conectado que puedas con los datos, guardando tu mejor modelo en la variable `best_model` y el solver que usaste en la variable `best_solver`. Se exige alcanzar al menos un 50% de precisión en el conjunto de validación con una red totalmente conectada.

Si tienes cuidado, debería ser posible superar el 55%, pero no se exige en esta parte y no habrá puntos extra. Más adelante en el ejercicio original se pedía entrenar la mejor red convolucional posible con CIFAR-10, y preferíamos que dedicaras tu esfuerzo a las redes convolucionales antes que a las totalmente conectadas.

> **Nota de la adaptación.** El texto original habla de CIFAR-10, pero los datos que carga este notebook son de Fashion-MNIST. Los umbrales del 50% y del 55% son los del enunciado original y pueden no corresponder a estos datos: úsalos como referencia, no como meta fija. La parte de redes convolucionales que menciona el texto no forma parte de este notebook.

In [ ]:
best_model = None
best_solver = None
################################################################################
# TODO: Entrena la mejor FullyConnectedNet que puedas. Puede que la            #
# normalización por lotes y el dropout te resulten útiles. Guarda tu mejor     #
# modelo en la variable best_model y el solver que usaste para entrenarlo en   #
# la variable best_solver.                                                     #
################################################################################

################################################################################
#                              FIN DE TU CÓDIGO                                #
################################################################################

# Prueba tu modelo
Ejecuta tu mejor modelo en los conjuntos de validación y de prueba, y guarda los registros de entrenamiento del mejor solver. Deberías alcanzar más de un 50% de precisión en el conjunto de validación.

In [ ]:
y_test_pred = np.argmax(best_model.loss(data['X_test']), axis=1)
y_val_pred = np.argmax(best_model.loss(data['X_val']), axis=1)
val_acc = (y_val_pred == data['y_val']).mean()
test_acc = (y_test_pred == data['y_test']).mean()
print ('Precisión en el conjunto de validación: ', val_acc)
print ('Precisión en el conjunto de prueba: ', test_acc)